# Custom interface spaces: own the basis, numbering and orientation

The normal path lets PyMHM derive representation details from a mesh-associated
interface. You can also supply every detail yourself through `InterfaceSpace`.
No inheritance or registration is needed: provide `size` and `binding(cell)`.

We use the affine Darcy pressure $p=1+x+2y$, identity permeability and zero
source on the unit square. The physical flux is $q=(-1,-2)$. First solve with
canonical degree-one Legendre face coordinates; then describe the same trace
space with a nonorthogonal basis and reversed global numbering. Pressure and
physical flux must be unchanged, although the interface coefficients change.

This is an analytical representation check, not a convergence study or a
reproduction of a published application. The local/global MHM decomposition is
described by [Harder and Valentin (2016)](https://doi.org/10.1007/978-3-319-41640-3_13).


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from threadpoolctl import threadpool_limits
from pymhm import (
    CartesianMacroMesh, Equation, FaceSpace, LocalContext, MeshHierarchy,
    SkeletonSpace, TraceBinding, bind_interface, bind_problem, assemble,
)
from pymhm.fem.scalar.quadrilateral import (
    qk_space, quadrilateral_operators, quadrilateral_trace_coupling,
)
from pymhm.fem.scalar.operators import boundary_data

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pixi.toml").exists())
OUTPUT = ROOT / "build/custom-interface"
OUTPUT.mkdir(parents=True, exist_ok=True)
threadpool_limits(1)

macro = CartesianMacroMesh(2, 2)
hierarchy = MeshHierarchy(macro, tuple(macro.submesh(cell, 4) for cell in range(len(macro.cells))))
skeleton = SkeletonSpace(macro, tuple(FaceSpace.uniform(1) for _ in macro.faces))


def exact_pressure(points: np.ndarray) -> np.ndarray:
    """Evaluate the affine pressure; the independently differentiated source is zero."""
    return 1 + points[:, 0] + 2 * points[:, 1]


## 1. Declare the local and global equations

The local equations retain the constant Neumann kernel through a physical
volume moment. We declare the two pairings independently:

$$
\begin{aligned}
(\nabla p_K,\nabla v)_K+\langle\lambda_K,v\rangle_{\partial K}&=0,\\
-\sum_K\langle p_K,\mu_K\rangle_{\partial K}&=-\langle p_D,\mu\rangle_{\partial\Omega}.
\end{aligned}
$$

The prepared coupling below already includes canonical normal signs. To write
blocks in outward-normal local coordinates, undo those signs once, then let
the declared interface binding transport the coordinates. This conversion is
explicit here because this is the advanced custom-space example. The introductory
UFL path uses `local.trace_pairings` to create unsigned local pairings directly.


In [ ]:
def local_equations(local: LocalContext):
    """Declare primal energy, physical flux pairing and the constant volume moment."""
    cell, fine = local.cell, local.mesh
    a, mass, load = quadrilateral_operators(fine, 2, source=0.0, order=5)
    canonical_b = quadrilateral_trace_coupling(macro, cell, fine, skeleton, 2)
    signs = np.repeat(macro.signs[cell], 2)
    outward_b = canonical_b * signs
    constant = np.ones((len(load), 1))
    return local.equations(
        a=a, L=load, b=outward_b, c=-outward_b.T,
        kernel=constant, moments=mass @ constant, metadata={"mesh": fine},
    )

boundary, _ = boundary_data(skeleton, exact_pressure, order=5)
canonical_problem = bind_problem(
    hierarchy, bind_interface(skeleton, convention="normal"), local_equations,
    global_equation=lambda global_problem: Equation(0, global_problem.trace_load(-boundary)),
    retained=1,
)
canonical_system = assemble(canonical_problem)
canonical_solution = canonical_system.solve()


## 2. Define the custom interface

On each face let $\varphi=(1,2t-1)$ and choose the nonorthogonal basis
$\psi=\varphi S$. Its coefficients $\widehat\lambda_F$ satisfy
$\lambda_F=S\widehat\lambda_F$ in canonical coordinates. We also choose our own
global numbering. The local binding combines that numbering, $S$, and the
normal incidence sign. Both trial and test coefficient maps are declared.

$$
\lambda_K=T_K\widehat\lambda[\mathrm{dofs}_K],\qquad
T_K=\operatorname{diag}_{F\subset\partial K}(s_{KF}S).
$$

`TraceBinding` checks finite dimensions and the requested injectivity and owns
immutable copies of the maps. Its digest identifies the declared numerical
basis and maps. It does not establish stability of a mathematical method.


In [ ]:
class CustomInterface:
    """Describe the same face space through custom numbering and dense basis maps."""

    def __init__(self, mesh: CartesianMacroMesh, change: np.ndarray) -> None:
        """Declare a complete invertible face-coordinate change and numbering."""
        self.mesh = mesh
        self.change = np.asarray(change, dtype=float).copy()
        if self.change.shape != (2, 2) or np.linalg.matrix_rank(self.change) != 2:
            raise ValueError("choose an invertible two-coordinate face basis")
        self.numbering = np.arange(self.size)[::-1].copy()

    @property
    def size(self) -> int:
        """Count two independent coefficients on every global macroface."""
        return 2 * len(self.mesh.faces)

    def binding(self, cell: int) -> TraceBinding:
        """Declare local supports, global coordinates and independent basis transports."""
        canonical_ids = np.concatenate([
            2 * int(face) + np.arange(2) for face in self.mesh.cell_faces[cell]
        ])
        transform = sparse.block_diag([
            sign * self.change for sign in self.mesh.signs[cell]
        ]).toarray()
        return TraceBinding(
            dofs=self.numbering[canonical_ids], trial_map=transform,
            test_dofs=self.numbering[canonical_ids], test_map=transform,
            basis_id="psi=(1,2t-1)S; reversed numbering; outward normal",
            require_injective=True,
        )

    def canonical_map(self) -> np.ndarray:
        """Map all custom coefficients to the independently declared canonical basis."""
        matrix = np.zeros((self.size, self.size))
        for face in range(len(self.mesh.faces)):
            ids = 2 * face + np.arange(2)
            matrix[np.ix_(ids, self.numbering[ids])] = self.change
        return matrix


S = np.array([[1.0, 0.4], [0.2, 1.1]])
custom = CustomInterface(macro, S)
P = custom.canonical_map()
custom_problem = bind_problem(
    hierarchy, custom, local_equations,
    global_equation=lambda global_problem: Equation(0, global_problem.trace_load(-P.T @ boundary)),
    retained=1,
)
custom_system = assemble(custom_problem)
custom_solution = custom_system.solve()


## 3. Check the same mathematical problem in both representations

Transport test equations with the transpose map, so the boundary load is
$P^Tg$ and the complete reduced operator changes by congruence. Retained pressure
means use their original coordinates. Comparing raw trace vectors would compare
different representations; compare transported traces and physical fields.


In [ ]:
transport = sparse.block_diag((P, np.eye(len(macro.cells)))).toarray()
np.testing.assert_allclose(
    custom_system.matrix.toarray(), transport.T @ canonical_system.matrix.toarray() @ transport,
    rtol=1e-10, atol=1e-10,
)
np.testing.assert_allclose(custom_system.rhs, transport.T @ canonical_system.rhs, rtol=1e-10, atol=1e-10)
np.testing.assert_allclose(P @ custom_solution.trace, canonical_solution.trace, rtol=1e-10, atol=1e-10)
field_error = 0.0
for cell, (old, new) in enumerate(zip(canonical_solution.fields, custom_solution.fields, strict=True)):
    nodes = qk_space(hierarchy.local_mesh(cell), 2)[1]
    np.testing.assert_allclose(new, old, rtol=1e-10, atol=1e-10)
    np.testing.assert_allclose(new, exact_pressure(nodes), rtol=1e-10, atol=1e-10)
    field_error = max(field_error, float(np.max(abs(new - exact_pressure(nodes)))))
print({"maximum_pressure_error": field_error,
       "canonical_physical_residual": canonical_solution.raw_residual,
       "custom_physical_residual": custom_solution.raw_residual,
       "custom_binding_digest": custom.binding(0).basis_digest})


## 4. Inspect the basis and the reconstructed pressure

The custom basis mixes constant and linear modes. Its span and the pressure
field are unchanged. Each field panel displays the actual macro mesh.


In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
t = np.linspace(0, 1, 101)
phi = skeleton.faces[0].evaluate(t)
for index in range(2):
    axes[0].plot(t, (phi @ S)[:, index], label=fr"$\psi_{index}$")
axes[0].set(xlabel="Face parameter t", ylabel="Basis value", title="Custom nonorthogonal face basis")
axes[0].legend()
for axis, title, fields in zip(axes[1:], ("Canonical pressure", "Custom-space pressure"),
                              (canonical_solution.fields, custom_solution.fields), strict=True):
    for cell, values in enumerate(fields):
        nodes = qk_space(hierarchy.local_mesh(cell), 2)[1]
        image = axis.scatter(nodes[:, 0], nodes[:, 1], c=values, vmin=1, vmax=4,
                             cmap="viridis", s=24)
    for endpoints in macro.points[macro.faces]:
        axis.plot(endpoints[:, 0], endpoints[:, 1], color="black", lw=0.7)
    axis.set(xlabel="x", ylabel="y", title=title, aspect="equal")
    figure.colorbar(image, ax=axis, label="Pressure")
figure.savefig(OUTPUT / "custom-interface.png", dpi=160)
plt.show()


## What else can a custom space provide?

A custom adapter can supply different trial and test maps, rectangular
restrictions, tangential rotations, external coefficient conventions and a
`trace_pairings(context, expression, axis=...)` capability for native forms.
Declare its mathematical support and meaning explicitly. Unsupported native
integration capabilities fail instead of guessing a cross-mesh coupling.

For complete manual control, `LocalEquations` and `MultiscaleProblem` remain
available. They use the same condensation, ordered assembly and solver owners as
`bind_problem`. Process execution requires portable picklable declarations and
worker-created native resources; the custom-space protocol does not change that
requirement. Existing multilevel and backend restrictions also remain in force.
